# Inicios


In [1]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path.cwd().parents[0] / "src"))
import numpy as np

# Conectando ao phonex


In [2]:
import os
from phoenix.otel import register
import phoenix as px
from openinference.instrumentation.langchain import LangChainInstrumentor

tracer_provider = register(
  project_name="Agente-Criador-Carteira",
  endpoint="https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces",
  auto_instrument=True,
  api_key=os.getenv("PHOENIX_API_KEY")
  
)

LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\Agent-Portfolio-Optimizer\.venv\Lib\site-packages\phoenix\otel\otel.py:434: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")
Attempting to instrument while already instrumented


OpenTelemetry Tracing Details
|  Phoenix Project: Agente-Criador-Carteira
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/sehnemjeferson/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



# Chamando agentes

In [47]:
from portfolio_optimizer import BuildGraphAvaliacaoTics, BuildGraphCriadorCarteira, StateClassification, TratatandoDadosFundamentalistas, DadosFundamentalistas, transformando_data_frame_para_markdown, StateCarteira


In [4]:
graph_tics = BuildGraphAvaliacaoTics()

In [5]:
graph_tics_build = graph_tics.compile()

In [22]:
response_tic = await graph_tics_build.ainvoke(
    StateClassification(
        {
            "tic": "ABEV3",
            "data_inicio": "2024-01-01",
            "data_fim": "2025-12-31",
        }
    )
)

2025-12-18 10:14:25,738 - INFO - Iniciando roteamento LLM
2025-12-18 10:14:25,738 - INFO - 🔄 Tentando Nvidia...
2025-12-18 10:14:45,696 - INFO - Sucesso com modelo Nvidia: qwen/qwq-32b
2025-12-18 10:14:45,696 - INFO - ✅ Nvidia respondeu com sucesso
2025-12-18 10:14:45,876 - INFO - Iniciando roteamento LLM
2025-12-18 10:14:45,876 - INFO - 🔄 Tentando Nvidia...
2025-12-18 10:15:10,854 - INFO - Sucesso com modelo Nvidia: qwen/qwq-32b
2025-12-18 10:15:10,855 - INFO - ✅ Nvidia respondeu com sucesso


In [29]:
response_tic

{'classification': 'Good',
 'analysis': 'ABEV3 shows stable revenue and improving margins despite debt. EBIDTA growth year-over-year, but leverage remains elevated. P/E and P/BV ratios indicate fair valuation. Cash flow volatility needs monitoring.',
 'tic': 'ABEV3',
 'dados_fundamentalistas': '|    |   receita_liquida |   ebitda |   lucro_por_acao | datas               | tic   |   alavancagem_financeira |   margem_liquida |   preço_lucro |   preço_vpa |   fluxo_caixa_operacional |   divida_liquida_ebitda |   aumento_reducao_caixa_equivalentes |\n|---:|------------------:|---------:|-----------------:|:--------------------|:------|-------------------------:|-----------------:|--------------:|------------:|--------------------------:|------------------------:|-------------------------------------:|\n|  1 |             20280 |     6450 |             0.23 | 2024-06-30 00:00:00 | ABEV3 |                     1.55 |            0.182 |         13.22 |        2.21 |                     718.2 |

In [42]:
from typing import List
import yfinance as yf
def transformando_data_frame_para_markdown(results):
    try:
        results_pd = pd.DataFrame.from_dict(results).T
    except Exception as e:
        results_pd = pd.DataFrame.from_dict(results, orient="index").T
    dados_markdown = results_pd.loc[:, ["classification", "analysis"]].reset_index().rename(columns={"index": "tic"}).to_markdown()
    return dados_markdown

def correlacao(tics:List[str]):
        tics_yf = [tic + ".SA" for tic in tics]

        df = yf.download(tics_yf, start="2023-01-01", end="2025-01-01", interval="1mo")["Close"]
        returns = df.pct_change()[4:]
        # Calcula a matriz de correlação
        correlacao = returns.corr()

        return correlacao.to_markdown()

In [19]:
import json
with open("avaliacao_acoes.json", "r") as f:
    response = json.load(f)

In [43]:
response_markdown = transformando_data_frame_para_markdown(response)

In [44]:
corr_tics = correlacao(list(response.keys()))

[*********************100%***********************]  10 of 10 completed


In [45]:
corr_tics

'| Ticker   |   ABEV3.SA |   BBAS3.SA |   BBDC4.SA |   GGBR4.SA |   ITUB4.SA |    LREN3.SA |   MGLU3.SA |    PETR4.SA |   RENT3.SA |    VALE3.SA |\n|:---------|-----------:|-----------:|-----------:|-----------:|-----------:|------------:|-----------:|------------:|-----------:|------------:|\n| ABEV3.SA |  1         |  0.63875   |  0.798136  |  0.294023  |  0.653014  |  0.765922   | 0.526548   |  0.260241   |   0.510053 |  0.0391894  |\n| BBAS3.SA |  0.63875   |  1         |  0.662714  | -0.0405815 |  0.769083  |  0.649864   | 0.531669   |  0.291555   |   0.559948 |  0.126629   |\n| BBDC4.SA |  0.798136  |  0.662714  |  1         |  0.18058   |  0.722328  |  0.890005   | 0.56217    |  0.335969   |   0.578477 |  0.0607044  |\n| GGBR4.SA |  0.294023  | -0.0405815 |  0.18058   |  1         |  0.298853  |  0.108999   | 0.276683   |  0.130243   |   0.366519 |  0.450482   |\n| ITUB4.SA |  0.653014  |  0.769083  |  0.722328  |  0.298853  |  1         |  0.766774   | 0.705956   |  0.0120056  

In [46]:
gaph_weights = BuildGraphCriadorCarteira()
graph_weights_build = gaph_weights.compile()

In [48]:
tics = list(response.keys())
tics

['VALE3',
 'PETR4',
 'ITUB4',
 'BBDC4',
 'ABEV3',
 'MGLU3',
 'BBAS3',
 'GGBR4',
 'RENT3',
 'LREN3']

In [ ]:
response_pesos = await graph_weights_build.ainvoke(
    StateCarteira({
    "justification" : "",
    "avaliacao_acoes": response_markdown ,
    "correlacao_acoes": corr_tics,
    "interacao" : 0,
    "tics": tics
    }))

2025-12-18 10:44:34,466 - INFO - Iniciando roteamento LLM
2025-12-18 10:44:34,468 - INFO - 🔄 Tentando Nvidia...
2025-12-18 10:44:53,863 - INFO - Sucesso com modelo Nvidia: qwen/qwq-32b
2025-12-18 10:44:53,865 - INFO - ✅ Nvidia respondeu com sucesso
2025-12-18 10:44:54,031 - INFO - ✓ Pesos já somam 100.00% (dentro da tolerância)
2025-12-18 10:44:54,032 - INFO - ✓ Pesos normalizados com sucesso: soma = 100.0000%
2025-12-18 10:44:54,187 - INFO - Os seguintes tickers não foram encontrados: Rental3
2025-12-18 10:44:54,559 - INFO - Iniciando roteamento LLM
2025-12-18 10:44:54,561 - INFO - 🔄 Tentando Nvidia...
2025-12-18 10:45:31,815 - INFO - Sucesso com modelo Nvidia: qwen/qwq-32b
2025-12-18 10:45:31,815 - INFO - ✅ Nvidia respondeu com sucesso
2025-12-18 10:45:31,998 - INFO - Iniciando roteamento LLM
2025-12-18 10:45:31,999 - INFO - 🔄 Tentando Nvidia...
2025-12-18 10:45:58,469 - INFO - Sucesso com modelo Nvidia: qwen/qwq-32b
2025-12-18 10:45:58,469 - INFO - ✅ Nvidia respondeu com sucesso
202